# Notebook 2 — Gradient descent, line search, and momentum

**Week 2 · Day 2 · ≈ 30 min, after Labwork 2**

> Yesterday's Figure 3 predicted a zigzag. Today you built the loop that does it, so now
> we can watch it happen and **measure** whether the predicted rate is the rate you get.

Needs, from day 2: `DescentOptimizer`, `SteepestDescent`, `HeavyBall`, `FixedStep`,
`Armijo`, `GradientNormBelow` / `MaxIterations` / `AnyOf`, and `History`.

| Figure | Shows |
|---|---|
| 1 | the zigzag itself, at three condition numbers |
| 2 | $\|\nabla f\|$ on a log scale — the rate, read off a straight line |
| 3 | what Armijo actually chose, step by step |
| 4 | the three regimes of a fixed step: converge, oscillate, diverge |
| 5 | $w$ reshaped to an 8×8 image — the parameters *are* a picture |

Throughout, note what the notebook never does: it never reaches inside the loop. Every
number below comes from `History`, an `IObserver` that the optimizer does not know is
there. That separation is the day's SOLID lesson, and this notebook is its payoff.

## 0. Setup

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# The optlab root is the nearest ancestor holding pyproject.toml, so this works whether
# Jupyter was started in notebooks/ or in the repository root.
HERE = Path.cwd()
ROOT = next((p for p in (HERE, *HERE.parents) if (p / "pyproject.toml").exists()), HERE.parent)

try:
    import optlab
except ModuleNotFoundError:
    sys.path.insert(0, str(ROOT / "src"))
    import optlab

sys.path.insert(0, str(ROOT))  # for datasets/

np.set_printoptions(precision=6, suppress=True)
plt.rcParams.update({"axes.grid": True, "grid.alpha": 0.3, "font.size": 10,
                     "axes.titlesize": 10, "figure.dpi": 110})
rng = np.random.default_rng(20250921)

print("optlab root  :", ROOT)
print("optlab loaded:", Path(optlab.__file__).parent)

from optlab.linesearch import Armijo, FixedStep                              # noqa: E402
from optlab.observers import History                                         # noqa: E402
from optlab.optimizers import DescentOptimizer, gradient_descent             # noqa: E402
from optlab.optimizers.directions import HeavyBall, SteepestDescent          # noqa: E402
from optlab.problems import Quadratic, logistic_regression                   # noqa: E402
from optlab.stopping import AnyOf, GradientNormBelow, MaxIterations          # noqa: E402

In [ ]:
def status(name, thunk):
    """Report whether one piece of the package is implemented, without a traceback."""
    try:
        thunk()
    except NotImplementedError:
        return f"  MISSING   {name}"
    except Exception as err:                      # noqa: BLE001 - we want to see anything
        return f"  BROKEN    {name}   ({type(err).__name__}: {err})"
    return f"  ok        {name}"

q2 = Quadratic.ill_conditioned(2, 10.0)
x1 = np.array([1.0, 1.0])

needed = [
    ("SteepestDescent",  lambda: SteepestDescent().direction(q2, x1, q2.gradient(x1))),
    ("HeavyBall",        lambda: HeavyBall().direction(q2, x1, q2.gradient(x1))),
    ("FixedStep",        lambda: FixedStep(0.1).step(q2, x1, q2.gradient(x1), -q2.gradient(x1))),
    ("Armijo",           lambda: Armijo().step(q2, x1, q2.gradient(x1), -q2.gradient(x1))),
    ("stopping criteria", lambda: AnyOf(GradientNormBelow(1e-6), MaxIterations(10))),
    ("History",          lambda: History().on_step(
        __import__("optlab").StepEvent(1, x1, 0.0, 0.0, 1.0))),
    ("DescentOptimizer", lambda: DescentOptimizer(
        SteepestDescent(), FixedStep(0.05), MaxIterations(3)).minimize(q2, x1)),
]

print("What this notebook needs from day 2:\n")
print("\n".join(status(name, thunk) for name, thunk in needed))

One helper, used everywhere below. It wires up a `DescentOptimizer`, attaches a fresh
`History`, runs it, and hands back both — the composition of day 2 in five lines.

In [ ]:
def run(direction, line_search, problem, x0, max_iter=400, tol=1e-12):
    """Compose an optimizer, run it, and return (result, history)."""
    hist = History()
    opt = DescentOptimizer(
        direction, line_search,
        AnyOf(GradientNormBelow(tol), MaxIterations(max_iter)),
        observers=[hist],
    )
    return opt.minimize(problem, np.asarray(x0, dtype=float)), hist


def path_of(hist, x0):
    """The iterates, as an array — read from the events, not from inside the loop."""
    return np.vstack([np.asarray(x0, dtype=float)] + [e.x for e in hist.events])

---

## 1. The zigzag

Steepest descent moves along $-\nabla f$, and yesterday's Figure 3 showed that this is
*not* the direction of the minimum unless the level sets are circles. Here is what that
costs, over the same problem at three condition numbers, from the same starting point,
with the same line search.

In [ ]:
kappas = [1.0, 10.0, 50.0]
fig, ax = plt.subplots(1, 3, figsize=(12, 4.0))
summary = []

for a, kappa in zip(ax, kappas, strict=True):
    q = Quadratic.ill_conditioned(2, kappa)
    x0 = np.array([1.0, 1.0 / np.sqrt(kappa)])          # same level set in every panel
    res, hist = run(SteepestDescent(), Armijo(), q, x0, max_iter=800, tol=1e-8)
    xs = path_of(hist, x0)

    g = np.linspace(-1.25, 1.25, 141)
    Z = np.array([[q.value(np.array([p, r])) for p in g] for r in g])
    a.contour(*np.meshgrid(g, g), Z, levels=np.geomspace(1e-3, Z.max(), 12),
              linewidths=0.8, alpha=0.65)
    a.plot(xs[:, 0], xs[:, 1], "o-", ms=3, lw=1.2, color="tab:red")
    a.plot(0, 0, "k*", ms=13)
    a.set_aspect("equal"), a.set_xlim(-1.25, 1.25), a.set_ylim(-1.25, 1.25)
    a.set_title(f"$\\kappa$ = {kappa:g}   —   {res.iterations} iterations")
    summary.append((kappa, res.iterations, res.converged))

fig.tight_layout()
plt.show()

print(f"{'kappa':>8}  {'iterations to 1e-8':>20}  converged")
for kappa, iters, ok in summary:
    print(f"{kappa:8g}  {iters:>20d}  {ok}")

**Figure 1 — the same algorithm, three difficulties.**

At $\kappa = 1$ the first step lands on the minimum: $-\nabla f$ points exactly at it, and
the line search only has to find the distance. As $\kappa$ grows the path develops the
characteristic **right-angle zigzag** — each step is orthogonal to the last, because an
exact-ish line search stops where the directional derivative vanishes, which is where the
new gradient is perpendicular to the direction just travelled.

The iteration counts in the table are the real message. Nothing about the algorithm
changed between the three panels; only the *problem* did. An optimizer's reputation is
inseparable from the conditioning of what it is asked to solve — which is why yesterday's
notebook spent a section on standardizing features.

---

## 2. Reading the rate off a straight line

On a quadratic, every method here converges **linearly**: the error is multiplied by a
constant factor $r$ each step. Plot $\|\nabla f\|$ on a log scale and linear convergence
becomes a straight line whose slope is $\log r$ — so the rate is something you can read
off the picture, and then check against theory.

Three predictions are on the table, and they are not the same:

| method | step | predicted factor |
|---|---|---|
| steepest descent | $\alpha = 1/L$ (the safe choice) | $1 - 1/\kappa$ |
| steepest descent | $\alpha = 2/(\lambda_{\min}+\lambda_{\max})$ (the best fixed step) | $(\kappa-1)/(\kappa+1)$ |
| heavy ball | the optimal pair $(\alpha, \beta)$ | $(\sqrt{\kappa}-1)/(\sqrt{\kappa}+1)$ |

The third is the one that matters: a **square root** of the condition number.

In [ ]:
kappa = 100.0
q = Quadratic.ill_conditioned(2, kappa)
lmin, lmax = 1.0, kappa                      # by construction of ill_conditioned
x0 = np.array([1.0, 1.0])

alpha_safe = 1.0 / lmax
alpha_best = 2.0 / (lmin + lmax)
beta_hb = ((np.sqrt(kappa) - 1) / (np.sqrt(kappa) + 1)) ** 2
alpha_hb = 4.0 / (np.sqrt(lmin) + np.sqrt(lmax)) ** 2

# (label for the legend, plain name for the table, direction, line search, prediction)
runs = [
    ("GD, $\\alpha = 1/L$", "GD, alpha = 1/L",
     SteepestDescent(), FixedStep(alpha_safe), 1 - 1 / kappa),
    ("GD, best fixed $\\alpha$", "GD, best fixed alpha",
     SteepestDescent(), FixedStep(alpha_best), (kappa - 1) / (kappa + 1)),
    ("GD + Armijo", "GD + Armijo",
     SteepestDescent(), Armijo(), None),
    ("heavy ball, optimal", "heavy ball, optimal",
     HeavyBall(beta_hb), FixedStep(alpha_hb), (np.sqrt(kappa) - 1) / (np.sqrt(kappa) + 1)),
]


def measured_factor(hist, tail=150):
    """Geometric mean of successive ||grad|| ratios over the tail of the run."""
    gn = np.array([e.grad_norm for e in hist.events])
    gn = gn[gn > 1e-13]
    seg = gn[-tail:] if len(gn) > tail else gn
    return float((seg[-1] / seg[0]) ** (1.0 / (len(seg) - 1)))


plt.figure(figsize=(7.2, 4.4))
print(f"{'method':<24} {'measured':>10} {'predicted':>11}   iterations to 1e-6")
for label, plain, direction, ls, predicted in runs:
    res, hist = run(direction, ls, q, x0, max_iter=2500, tol=1e-10)
    gn = [e.grad_norm for e in hist.events]
    plt.semilogy(gn, lw=1.6, label=label)
    hit = next((i + 1 for i, v in enumerate(gn) if v <= 1e-6), None)
    pred = f"{predicted:.6f}" if predicted is not None else "        --"
    print(f"{plain:<24} {measured_factor(hist):10.6f} {pred:>11}   {hit}")

plt.xlabel("iteration"), plt.ylabel("$\\|\\nabla f\\|$")
plt.title(f"convergence on the quadratic with $\\kappa$ = {kappa:g}")
plt.legend(fontsize=8), plt.ylim(1e-11, None), plt.tight_layout()
plt.show()

**Figure 2 — four straight lines, four different slopes.**

Every curve is straight on this log axis, which *is* the statement "convergence is
linear". What differs is the slope, and the printed table compares each measured factor
against its prediction.

- The two fixed-step gradient-descent rows match their predictions to six decimal places.
  That is not a coincidence or a fit: on a quadratic the error contracts by exactly
  $\max_i|1 - \alpha\lambda_i|$, and both rows are that formula evaluated at their $\alpha$.
- The gap between the two is worth pausing on. $\alpha = 1/L$ is the step the descent
  lemma guarantees is *safe* for any $L$-smooth function; the best fixed step for this
  particular quadratic is nearly twice as large. Safety costs iterations.
- **Armijo does slightly better than the best fixed step**, which at first looks
  impossible — the "best fixed step" is optimal among *constant* steps, and Armijo is not
  constant. It re-chooses $\alpha$ every iteration, so it is competing in a larger class.
- Heavy ball is in a different league, and its measured factor sits near but not exactly
  on $(\sqrt{\kappa}-1)/(\sqrt{\kappa}+1)$. That prediction is **asymptotic** — it
  describes the limiting behaviour of the two-term recursion, and a short run measured
  over a finite window does not have to reproduce it to the digit. What is not in doubt
  is the scaling: the count of iterations to $10^{-6}$ falls by roughly the square root.

---

## 3. What the line search actually did

`StepEvent` carries `step_size`, so the history records the $\alpha$ chosen at every
iteration. Nothing had to be added to the loop to collect this — a second observer would
have seen exactly the same stream.

Two problems, side by side, because they behave completely differently: the quadratic
from Figure 2, whose curvature is the same everywhere, and **Rosenbrock**, whose curvature
changes enormously along the valley.

In [ ]:
from optlab.problems import Rosenbrock                                       # noqa: E402

_, hist_q = run(SteepestDescent(), Armijo(), q, x0, max_iter=250, tol=1e-10)
_, hist_r = run(SteepestDescent(), Armijo(), Rosenbrock(), np.array([-1.2, 1.0]),
                max_iter=400, tol=1e-8)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
for a, hist, title in [(ax[0], hist_q, f"quadratic, $\\kappa$ = {kappa:g}"),
                       (ax[1], hist_r, "Rosenbrock")]:
    steps = [e.step_size for e in hist.events]
    a.semilogy(steps, lw=1.2, color="tab:red")
    a.set_xlabel("iteration"), a.set_ylabel("step length $\\alpha$")
    a.set_title(f"{title} — {len(np.unique(np.round(steps, 12)))} distinct values")
ax[0].axhline(alpha_safe, color="tab:blue", ls="--", lw=1.2, label="$1/L$")
ax[0].axhline(alpha_best, color="tab:green", ls=":", lw=1.4, label="best fixed step")
ax[0].legend(fontsize=8)
fig.tight_layout()
plt.show()

for name, hist in [("quadratic", hist_q), ("Rosenbrock", hist_r)]:
    vals = np.array([e.step_size for e in hist.events])
    uniq = np.unique(np.round(vals, 12))
    print(f"{name:<11} {len(uniq):>2} distinct steps, from {vals.min():.2e} to {vals.max():.2e}"
          f"   (all are 1.0 x 0.5^k: {np.allclose(np.log2(uniq) % 1, 0)})")
print(f"\nquadratic: fraction of steps above 1/L = {np.mean(np.array([e.step_size for e in hist_q.events]) > alpha_safe):.2f}")

**Figure 3 — backtracking is not a smooth dial, and how coarse it is depends on the problem.**

Every step length is $\alpha_0 \rho^k$ — here $1.0 \times 0.5^k$ — because that is
literally what backtracking produces: start at $\alpha_0$, halve until the sufficient
decrease condition holds. The printed check confirms every value taken is an exact power
of two.

*Left.* On the quadratic, Armijo finds its level almost immediately and then alternates
between **two** values for the rest of the run. Constant curvature means the right step is
the same step every time, so the line search has nothing to adapt to. Note also that every
one of those steps is **larger than $1/L$**: the descent lemma's bound is one that must
work without knowing anything about $f$ beyond $L$, while Armijo tests the actual decrease
at the actual point and accepts whatever passes.

*Right.* Rosenbrock spans several orders of magnitude of step length. Along the floor of
the curved valley the safe step is tiny; on the approach it is large. No constant would
have served, which is the honest argument for a line search: not that it is cleverer, but
that it needs no constant from you — and $L$ is rarely known for a real objective.

---

## 4. Break it: the three regimes of a fixed step

The descent lemma says $\alpha \le 1/L$ is safe. What happens on the other side of that
line is worth seeing rather than being told. For a quadratic the exact boundary is
known — the iteration $x^+ = x - \alpha A x$ contracts if and only if
$|1 - \alpha\lambda| < 1$ for every eigenvalue, i.e. $\alpha < 2/\lambda_{\max} = 2/L$.

So there are three regimes, and the middle one is the interesting one.

In [ ]:
q10 = Quadratic.ill_conditioned(2, 10.0)
L10 = 10.0
start = np.array([1.0, 1.0])

eigs = np.linalg.eigvalsh(q10.A)             # 1 and 10, by construction

cases = [
    ("$\\alpha = 0.5/L$  — safe",        "alpha = 0.5/L  safe",        0.5 / L10),
    ("$\\alpha = 1.0/L$  — the bound",   "alpha = 1.0/L  the bound",   1.0 / L10),
    ("$\\alpha = 1.9/L$  — still < 2/L", "alpha = 1.9/L  still < 2/L", 1.9 / L10),
    ("$\\alpha = 2.1/L$  — past 2/L",    "alpha = 2.1/L  past 2/L",    2.1 / L10),
]

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
print(f"{'case':<28} {'|1-a*1|':>9} {'|1-a*10|':>9} {'max':>8} {'final ||x||':>14}")
for label, plain, alpha in cases:
    _, hist = run(SteepestDescent(), FixedStep(alpha), q10, start, max_iter=60, tol=0.0)
    xs = path_of(hist, start)
    norms = np.linalg.norm(xs, axis=1)
    ax[0].semilogy(norms, lw=1.5, label=label)
    ax[1].plot(xs[:12, 0], xs[:12, 1], "o-", ms=3, lw=1.1, label=label)
    factors = np.abs(1 - alpha * eigs)
    print(f"{plain:<28} {factors[0]:>9.4f} {factors[1]:>9.4f} {factors.max():>8.4f}"
          f" {norms[-1]:>14.4e}")

ax[0].set_xlabel("iteration"), ax[0].set_ylabel("$\\|x_k\\|$")
ax[0].set_title("distance to the minimum"), ax[0].legend(fontsize=8)
ax[0].set_ylim(1e-8, 1e8)
ax[1].set_xlabel("$x_1$"), ax[1].set_ylabel("$x_2$"), ax[1].set_title("the first 12 iterates")
ax[1].set_xlim(-2, 2), ax[1].set_ylim(-2, 2), ax[1].axhline(0, color="k", lw=0.6)
fig.tight_layout()
plt.show()

**Figure 4 — the boundary is at $2/L$, not $1/L$.**

The printed table has one factor per eigenvalue, and the rate is the **largest** of them:
the error contracts by $\max_i |1 - \alpha\lambda_i|$ per step. Whenever that maximum is
below $1$ the run converges; when it exceeds $1$ the run explodes. Everything else in the
figure follows from that one column.

- $\alpha = 0.5/L$: convergence at $0.95$ per step, limited by the *soft* direction
  $\lambda = 1$, which the small step barely moves.
- $\alpha = 2.1/L$: divergence, and fast — $|1 - 2.1| = 1.1 > 1$, so the stiff direction
  grows by 10 % per iteration and the left panel climbs by orders of magnitude.
- **$\alpha = 1.0/L$ and $\alpha = 1.9/L$ converge at exactly the same speed**, which
  looks like a mistake until you read the two columns. At $\alpha = 1/L$ the stiff
  direction has factor $|1 - 1| = 0$ — it is annihilated in a *single* step — and the run
  is then governed by the soft direction at $0.9$. At $\alpha = 1.9/L$ the roles are
  swapped: the soft direction contracts nicely at $0.81$ while the stiff one limps along
  at $|1 - 1.9| = 0.9$. Same maximum, same rate, completely different reason. The right
  panel shows the difference in character — at $1.9/L$ the iterates **overshoot and flip
  sign** across the axis every step, because that factor is negative.

The moral is not "use a smaller step". It is that **$1/L$ is a sufficient condition, not
the truth**, and that what fails past $2/L$ is not numerical noise but an eigenvalue
crossing $-1$. This is the same calculation that will tell you, on day 3, why a stochastic
step size must eventually decay.

---

## 5. The parameters are a picture

Everything so far has been two-dimensional so it could be drawn. Real $w$ lives in
hundreds of dimensions — but when the features have a spatial layout, $w$ does too, and
you can simply *look* at it.

The day-2 dataset for this is the handwritten digits, 3 against 8, where each of the 64
features is one pixel of an 8×8 image. That set is not in this clone, so the cell below
builds a synthetic stand-in with the same shape and a **known** answer: two 8×8 templates,
noisy samples of each, and a logistic regression fitted with your `gradient_descent`. If
the fit works, $w$ must look like the difference of the two templates — and unlike with
real digits, we can check that claim exactly.

In [ ]:
side, n, noise = 8, 600, 1.5               # noise high enough that the classes overlap
tmpl_a, tmpl_b = np.zeros((side, side)), np.zeros((side, side))
tmpl_a[2:6, 3] = 1.0                       # a vertical bar
tmpl_b[3, 2:6] = 1.0                       # a horizontal bar

labels = (rng.random(n) < 0.5).astype(float)
images = np.array([(tmpl_b if y else tmpl_a) + noise * rng.normal(size=(side, side))
                   for y in labels])
X = images.reshape(n, side * side)

res, hist = run(SteepestDescent(), Armijo(), logistic_regression(X, labels),
                np.zeros(side * side), max_iter=800, tol=1e-6)
w_img = res.x.reshape(side, side)
truth = tmpl_b - tmpl_a                    # what w should be proportional to

fig, ax = plt.subplots(1, 3, figsize=(11, 3.4))
for a, (img, title) in zip(ax, [
    (images[labels == 0].mean(axis=0), "mean image, class 0"),
    (images[labels == 1].mean(axis=0), "mean image, class 1"),
    (w_img, "the fitted $w$, reshaped 8x8"),
], strict=True):
    lim = np.abs(img).max()
    im = a.imshow(img, cmap="RdBu_r", vmin=-lim, vmax=lim)
    a.set_title(title), a.set_xticks([]), a.set_yticks([]), a.grid(False)
    fig.colorbar(im, ax=a, fraction=0.046)
fig.tight_layout()
plt.show()

flat_w, flat_t = w_img.ravel(), truth.ravel()
cos = float(flat_w @ flat_t / (np.linalg.norm(flat_w) * np.linalg.norm(flat_t)))
pred = (X @ res.x > 0).astype(float)
print(f"iterations                      : {res.iterations}  (converged={res.converged})")
print(f"training accuracy               : {np.mean(pred == labels):.4f}")
print(f"cosine(w, template difference)  : {cos:.4f}")
print(f"largest |w| pixel               : {np.unravel_index(np.abs(w_img).argmax(), w_img.shape)}")
print(f"the two template pixels are     : rows 2-5 col 3 (class 0), row 3 cols 2-5 (class 1)")

**Figure 5 — a coefficient vector you can read.**

The third panel is the same numbers the optimizer has been printing all along, folded
back into the shape the data came in. Blue where a bright pixel argues for class 0, red
where it argues for class 1 — and it reproduces the two bars, because that is the only
thing that distinguishes the classes.

The printed cosine between $w$ and the true template difference says how close that
resemblance is in numbers rather than in impression. It is high but not $1$, and it should
not be $1$: logistic regression is free to scale $w$ however it likes (the loss only sees
$Xw$), and with $600$ heavily noised samples it also picks up some of the noise.

**Why this matters beyond the pretty picture.** When a fit goes wrong, a coefficient
vector is usually a wall of numbers that tells you nothing. Reshaped, it tells you
immediately whether the model learned the signal or an artefact — a border pixel, a
scanner edge, the patient ID accidentally left in column 3. Look at your parameters.

*If the real `digits` data is ever prepared into `data/`, rerun this section on
`load("digits_3v8")` — the picture is the same idea with a much more interesting shape.*

---

## Checkpoint

1. Why does the zigzag appear, and what makes consecutive steps orthogonal?
2. What exactly does $(\kappa-1)/(\kappa+1)$ predict, and for which step size?
3. Why can Armijo beat the best constant step?
4. Where is the true stability boundary for a fixed step, and what happens between $1/L$
   and $2/L$?
5. What did you have to change inside `DescentOptimizer` to produce any figure in this
   notebook? *(Nothing. That is the answer, and it is the point.)*

### Tomorrow

Day 3 replaces the exact gradient with a **sample** of it. The loop changes shape — epochs
and shuffling, no line search — so SGD and Adam are new `IOptimizer` implementations rather
than new `IDirectionRule`s. Figure 4's stability calculation comes straight back: with a
noisy gradient a constant step cannot converge to a point at all, only to a *noise floor*,
and you will measure it.